# Collect Cosmos-Policy contrastive input pairs: clean vs gaussian-noised images

Builds paired policy-model inputs for downstream contrastive-direction
analysis (LQR / SVD). Single prompt and a single clean rollout per episode
for `libero_10` task `0`; the negative inputs are constructed *post-hoc* by
adding i.i.d. Gaussian pixel noise to the captured wrist and 3rd-person
images. The noise level matches the rollouts under
`notebooks/stress_test/rollouts/libero_10__task00__img_noise_extreme`
(`ImageGaussianNoise(sigma=90.0, apply_to=(agentview_image, robot0_eye_in_hand_image), per_episode_seed=True)`).

Prompt (single, used everywhere):

  `"put both the alphabet soup and the tomato sauce in the basket"`

Pairing: row i in `positive.npz` and row i in `negative.npz` share the same
MuJoCo state (and therefore identical proprio); only the image content
differs. The negative images are the positive images plus Gaussian noise
(σ=90 in uint8 units, clipped to [0, 255]). Per-row tags
`(episode_idx, inference_idx, drive_source)` mirror the schema produced by
`collect_policy_inputs_milk.ipynb`, so the output plugs straight into
`run_partition_svd_pairs.py`. `drive_source` is fixed to 0 (single drive
campaign).

Output layout:

```
notebooks/lqr/inputs/policy_inputs/libero_10__task00__noise_extreme_pos_neg/
  positive.npz   # clean renders captured at each inference
  negative.npz   # positive + Gaussian noise on wrist + primary
  manifest.json  # prompt, noise config, per-rollout summaries
```

In [ ]:
import sys; sys.path.insert(0, '../..')
from _setup import setup_env
setup_env()

import os
os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')

In [ ]:
import json
import time
from collections import deque
from pathlib import Path

import numpy as np

from libero.libero import benchmark
from cosmos_policy.experiments.robot.libero.libero_utils import (
    get_libero_env, get_libero_dummy_action,
)
from cosmos_policy.experiments.robot.libero.run_libero_eval import (
    PolicyEvalConfig, prepare_observation, TASK_MAX_STEPS,
)
from cosmos_policy.experiments.robot.cosmos_utils import (
    get_action, get_model, load_dataset_stats, init_t5_text_embeddings_cache,
)

## 1. Config

Noise parameters mirror the `noise_extreme` preset under
`notebooks/stress_test/rollouts/libero_10__task00__img_noise_extreme/manifest.json`:
`sigma=90.0`, `apply_to=(agentview_image, robot0_eye_in_hand_image)`,
`per_episode_seed=True`.

In [ ]:
SUITE_NAME  = 'libero_10'
TASK_ID     = 0
N_EPISODES  = 10
RESOLUTION  = 256

PROMPT = 'put both the alphabet soup and the tomato sauce in the basket'

NOISE_SIGMA           = 75.0   # uint8 pixel units; matches noise_extreme
NOISE_PER_EPISODE_SEED = True   # seed = int(episode_idx), matches stress test

OUT_DIR = Path('notebooks/lqr/inputs/policy_inputs') / f'{SUITE_NAME}__task{TASK_ID:02d}__noise_high_4_pos_neg'
OUT_DIR.mkdir(parents=True, exist_ok=True)
POSITIVE_NPZ  = OUT_DIR / 'positive.npz'
NEGATIVE_NPZ  = OUT_DIR / 'negative.npz'
MANIFEST_JSON = OUT_DIR / 'manifest.json'

print(f'suite     : {SUITE_NAME}')
print(f'task      : {TASK_ID}')
print(f'episodes  : {N_EPISODES}')
print(f'prompt    : {PROMPT!r}')
print(f'noise     : sigma={NOISE_SIGMA}  per_episode_seed={NOISE_PER_EPISODE_SEED}')
print(f'positive -> {POSITIVE_NPZ.resolve()}')
print(f'negative -> {NEGATIVE_NPZ.resolve()}')

## 2. Build env + load saved initial states

Standard LIBERO env build — same as `collect_policy_inputs.ipynb`.

In [ ]:
task_suite = benchmark.get_benchmark_dict()[SUITE_NAME]()
task = task_suite.get_task(TASK_ID)
init_states = task_suite.get_task_init_states(TASK_ID)
print(f'task.language: {task.language!r}')
print(f'init states available: {init_states.shape[0]}  (using the first {N_EPISODES})')
assert N_EPISODES <= init_states.shape[0], f'only {init_states.shape[0]} init states available'

env, task_desc_from_env = get_libero_env(task, 'cosmos', resolution=RESOLUTION)
max_env_steps = TASK_MAX_STEPS[SUITE_NAME]
print(f'env={type(env).__name__}  max_steps={max_env_steps}')
print(f'env-provided task_desc: {task_desc_from_env!r}')

## 3. Load the Cosmos-Policy checkpoint

In [ ]:
cfg = PolicyEvalConfig(
    config='cosmos_predict2_2b_480p_libero__inference_only',
    ckpt_path='nvidia/Cosmos-Policy-LIBERO-Predict2-2B',
    config_file='cosmos_policy/config/config.py',
    dataset_stats_path='nvidia/Cosmos-Policy-LIBERO-Predict2-2B/libero_dataset_statistics.json',
    t5_text_embeddings_path='nvidia/Cosmos-Policy-LIBERO-Predict2-2B/libero_t5_embeddings.pkl',
    use_wrist_image=True, use_proprio=True, normalize_proprio=True, unnormalize_actions=True,
    chunk_size=16, num_open_loop_steps=16, trained_with_image_aug=True,
    use_jpeg_compression=True, flip_images=True,
    num_denoising_steps_action=5,
    num_denoising_steps_future_state=1, num_denoising_steps_value=1,
    task_suite_name=SUITE_NAME,
)
dataset_stats = load_dataset_stats(cfg.dataset_stats_path)
init_t5_text_embeddings_cache(cfg.t5_text_embeddings_path)
model, _ = get_model(cfg)
print('model ready')

## 4. Rollout that records every policy input

Clean rollout under the original task prompt. At each inference call we
snapshot the `prepare_observation` packed dict — `primary_image` (256×256×3
uint8, flipped), `wrist_image` (256×256×3 uint8, flipped), `proprio` (9-D
float32) — which is exactly the input `get_action` consumes on replay.

In [ ]:
def policy_fn(obs, desc):
    out = get_action(
        cfg, model, dataset_stats, obs, desc,
        num_denoising_steps_action=cfg.num_denoising_steps_action,
        generate_future_state_and_value_in_parallel=True,
    )
    return out['actions']


def rollout_collect_inputs(env, init_state, task_desc, *, num_steps_wait=10):
    env.reset()
    obs = env.set_init_state(init_state)

    for _ in range(num_steps_wait):
        obs, _, _, _ = env.step(get_libero_dummy_action(cfg.model_family))

    queue = deque(maxlen=cfg.num_open_loop_steps)
    inputs = []
    success = False
    t = 0
    while t < max_env_steps:
        if not queue:
            observation = prepare_observation(obs, resize_size=224, flip_images=cfg.flip_images)
            inputs.append({
                'primary_image': np.ascontiguousarray(observation['primary_image']),
                'wrist_image':   np.ascontiguousarray(observation['wrist_image']),
                'proprio':       np.asarray(observation['proprio'], dtype=np.float32),
            })
            actions = policy_fn(observation, task_desc)
            for a in actions[:cfg.num_open_loop_steps]:
                queue.append(np.asarray(a, dtype=np.float32))
        a = queue.popleft()
        obs, _, done, _ = env.step(a.tolist())
        if done:
            success = True
            break
        t += 1
    return success, t + num_steps_wait, inputs

## 5. Run all 10 clean rollouts and accumulate positive inputs

In [ ]:
all_primary, all_wrist, all_proprio = [], [], []
all_episode_idx, all_inference_idx = [], []
rollout_summaries = []

for ep in range(N_EPISODES):
    t0 = time.time()
    success, env_steps, inputs = rollout_collect_inputs(env, init_states[ep], PROMPT)
    dt = time.time() - t0

    for inf_idx, rec in enumerate(inputs):
        all_primary.append(rec['primary_image'])
        all_wrist.append(rec['wrist_image'])
        all_proprio.append(rec['proprio'])
        all_episode_idx.append(ep)
        all_inference_idx.append(inf_idx)

    tag = 'SUCCESS' if success else 'FAILURE'
    n_inf = len(inputs)
    print(f'ep {ep:2d}  {tag:7s}  steps={env_steps:4d}  inferences={n_inf:3d}  {dt:6.1f}s')
    rollout_summaries.append({
        'episode': ep,
        'success': bool(success),
        'env_steps': int(env_steps),
        'n_inferences': n_inf,
        'wall_time_s': dt,
    })

env.close()

pos_primary = np.stack(all_primary, axis=0)
pos_wrist   = np.stack(all_wrist,   axis=0)
proprio_arr = np.stack(all_proprio, axis=0)
episode_arr   = np.asarray(all_episode_idx,   dtype=np.int32)
inference_arr = np.asarray(all_inference_idx, dtype=np.int32)
drive_arr     = np.zeros_like(episode_arr, dtype=np.int32)

print()
print(f'total inferences collected: {pos_primary.shape[0]}')
print(f'  primary_images: {pos_primary.shape}  dtype={pos_primary.dtype}')
print(f'  wrist_images:   {pos_wrist.shape}  dtype={pos_wrist.dtype}')
print(f'  proprios:       {proprio_arr.shape}  dtype={proprio_arr.dtype}')

## 6. Construct negative inputs by adding Gaussian pixel noise

For each captured (primary, wrist) pair, add i.i.d. Gaussian noise
(σ=`NOISE_SIGMA` in uint8 units) and clip back to [0, 255] uint8. The RNG is
re-seeded at the start of each episode with `seed=int(episode_idx)` so the
noise pattern is reproducible across runs. Within an episode, noise is drawn
fresh for each captured (primary, wrist) pair, in inference order — same
iteration order the `ImageGaussianNoise` stress test would use under a clean
rollout (one draw per perturbed obs).

In [ ]:
neg_primary = np.empty_like(pos_primary)
neg_wrist   = np.empty_like(pos_wrist)

current_ep = None
rng = None
for row in range(pos_primary.shape[0]):
    ep = int(episode_arr[row])
    if NOISE_PER_EPISODE_SEED and ep != current_ep:
        rng = np.random.default_rng(seed=ep)
        current_ep = ep
    elif rng is None:
        rng = np.random.default_rng()

    p_img = pos_primary[row].astype(np.float32)
    p_noise = rng.normal(loc=0.0, scale=NOISE_SIGMA, size=p_img.shape).astype(np.float32)
    neg_primary[row] = np.clip(p_img + p_noise, 0, 255).astype(np.uint8)

    w_img = pos_wrist[row].astype(np.float32)
    w_noise = rng.normal(loc=0.0, scale=NOISE_SIGMA, size=w_img.shape).astype(np.float32)
    neg_wrist[row] = np.clip(w_img + w_noise, 0, 255).astype(np.uint8)

abs_diff_primary = np.abs(neg_primary.astype(np.float32) - pos_primary.astype(np.float32))
abs_diff_wrist   = np.abs(neg_wrist.astype(np.float32)   - pos_wrist.astype(np.float32))
print(f'primary |delta| mean={abs_diff_primary.mean():.2f}  max={abs_diff_primary.max():.2f}')
print(f'wrist   |delta| mean={abs_diff_wrist.mean():.2f}  max={abs_diff_wrist.max():.2f}')

## 7. Save paired NPZs and manifest

Both files share row count and the `(episode_idx, inference_idx, drive_source)`
columns. Row `i` in `positive.npz` and row `i` in `negative.npz` share the
same proprio by construction (negative is constructed from positive).

In [ ]:
np.savez_compressed(
    POSITIVE_NPZ,
    primary_images=pos_primary,
    wrist_images=pos_wrist,
    proprios=proprio_arr,
    episode_idx=episode_arr,
    inference_idx=inference_arr,
    drive_source=drive_arr,
)
size_mb_pos = POSITIVE_NPZ.stat().st_size / 1e6
print(f'wrote {POSITIVE_NPZ}  ({size_mb_pos:.1f} MB)')

np.savez_compressed(
    NEGATIVE_NPZ,
    primary_images=neg_primary,
    wrist_images=neg_wrist,
    proprios=proprio_arr,
    episode_idx=episode_arr,
    inference_idx=inference_arr,
    drive_source=drive_arr,
)
size_mb_neg = NEGATIVE_NPZ.stat().st_size / 1e6
print(f'wrote {NEGATIVE_NPZ}  ({size_mb_neg:.1f} MB)')

manifest = {
    'suite': SUITE_NAME,
    'task_id': TASK_ID,
    'n_episodes': N_EPISODES,
    'resolution': RESOLUTION,
    'prompt': PROMPT,
    'pairing': (
        'row i in positive.npz and row i in negative.npz share the same '
        'MuJoCo state at capture time. negative is constructed post-hoc by '
        'adding Gaussian noise to positive\'s primary_image and wrist_image. '
        'drive_source is 0 for all rows (single clean drive campaign).'
    ),
    'image_layout': 'HWC uint8, flip_images=True applied at capture time (same as get_action input)',
    'proprio_layout': 'concat(robot0_gripper_qpos[2], robot0_eef_pos[3], robot0_eef_quat[4]) -> shape (9,) float32',
    'noise': {
        'kind': 'ImageGaussianNoise',
        'sigma': float(NOISE_SIGMA),
        'apply_to': ['primary_image', 'wrist_image'],
        'per_episode_seed': bool(NOISE_PER_EPISODE_SEED),
        'seed_recipe': 'np.random.default_rng(seed=int(episode_idx)); within each episode, draw primary noise then wrist noise per inference (in inference order)',
        'clipped_to': [0, 255],
        'output_dtype': 'uint8',
        'matches_rollout_preset': 'noise_high_4 (notebooks/stress_test/rollouts/libero_10__task00__img_noise_high_4/manifest.json)',
    },
    'drive_sources': [
        {'code': 0, 'name': 'clean_drive', 'desc': 'env steps cleanly under PROMPT; negative is post-hoc noised'},
    ],
    'sets': {
        'positive': {'out_npz': str(POSITIVE_NPZ), 'role': 'clean renders at every captured pose'},
        'negative': {'out_npz': str(NEGATIVE_NPZ), 'role': 'positive + Gaussian noise on both cameras'},
    },
    'total_inferences': int(pos_primary.shape[0]),
    'rollouts': rollout_summaries,
    'paired_proprio_max_abs_diff': 0.0,
    'pixel_diff_stats': {
        'primary_abs_diff_mean': float(abs_diff_primary.mean()),
        'primary_abs_diff_max':  float(abs_diff_primary.max()),
        'wrist_abs_diff_mean':   float(abs_diff_wrist.mean()),
        'wrist_abs_diff_max':    float(abs_diff_wrist.max()),
    },
}
MANIFEST_JSON.write_text(json.dumps(manifest, indent=2))
print(f'wrote {MANIFEST_JSON}')

## 8. Visualize a few paired examples

Sanity-check the contrastive setup: a few evenly-spaced inference rows from
episode 0, side-by-side (clean | noised) for both the primary (agentview)
and wrist cameras. Within a row, the two scenes differ only in pixel noise
by construction.

In [ ]:
import matplotlib.pyplot as plt

VIZ_EPISODE = 0
VIZ_SAMPLES = 3

rows_in_ep = np.where(episode_arr == VIZ_EPISODE)[0]
if len(rows_in_ep) == 0:
    print(f'no rows for episode {VIZ_EPISODE}; nothing to visualize')
else:
    sel = rows_in_ep if len(rows_in_ep) <= VIZ_SAMPLES else rows_in_ep[
        np.linspace(0, len(rows_in_ep) - 1, VIZ_SAMPLES).astype(int)
    ]
    n_rows = len(sel)
    fig, axes = plt.subplots(n_rows, 4, figsize=(13, 3.2 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]
    for r_i, i in enumerate(sel):
        inf_idx = int(inference_arr[i])
        axes[r_i, 0].imshow(pos_primary[i]); axes[r_i, 0].set_title(f'ep{VIZ_EPISODE} inf{inf_idx:3d}\npos primary (clean)', fontsize=9); axes[r_i, 0].axis('off')
        axes[r_i, 1].imshow(neg_primary[i]); axes[r_i, 1].set_title(f'neg primary (σ={NOISE_SIGMA})', fontsize=9);                       axes[r_i, 1].axis('off')
        axes[r_i, 2].imshow(pos_wrist[i]);   axes[r_i, 2].set_title('pos wrist (clean)', fontsize=9);                                     axes[r_i, 2].axis('off')
        axes[r_i, 3].imshow(neg_wrist[i]);   axes[r_i, 3].set_title(f'neg wrist (σ={NOISE_SIGMA})', fontsize=9);                          axes[r_i, 3].axis('off')
    plt.tight_layout()
    plt.show()

## 9. Summary

In [ ]:
n_succ = sum(r['success'] for r in rollout_summaries)
n_inf  = sum(r['n_inferences'] for r in rollout_summaries)
print(f'  clean rollouts success    : {n_succ}/{N_EPISODES}')
print(f'  total paired rows         : {n_inf}')
print(f'  pixel |delta| primary mean: {abs_diff_primary.mean():.2f}')
print(f'  pixel |delta| wrist   mean: {abs_diff_wrist.mean():.2f}')
print(f'  positive (clean)          : {POSITIVE_NPZ.resolve()}')
print(f'  negative (noised)         : {NEGATIVE_NPZ.resolve()}')
print(f'  manifest                  : {MANIFEST_JSON.resolve()}')